# GraphRAG Pipeline — Financial Knowledge Graph
## FinBERT NER → SSHG → GCN → LLM Relation Extraction → Neo4j → Leiden → QA

**Pipeline overview (two parts):**

| Part | Steps |
|------|-------|
| **Part 1 – KG Creation (offline)** | Preprocessing → FinBERT+BIO NER → SSHG construction → GCN propagation → Temporal/Numeric anchoring → LLM relation extraction (Llama 3.2 3B) → Neo4j graph → Leiden communities |
| **Part 2 – Retrieval & QA (inference)** | Query type classification → Router (local/global/hybrid) → Mistral-NeMo answer generation → RAG Triad evaluation |

**Fairness vs baselines:** same 5 000-article Bloomberg corpus, same 20 questions from `qa_set.json`, same `llama3.2:3b` judge.

## 0. Environment Setup

Install all dependencies. Run once.

In [ ]:
# Core NLP + graph
!pip install transformers torch spacy networkx sentence-transformers -q
!pip install python-dateutil requests datasets tqdm rapidfuzz -q
# Supervised NER / RE fine-tuning
!pip install peft accelerate bitsandbytes -q
!pip install seqeval -q          # span-level NER evaluation (seqeval)
# Hyperparameter optimisation
!pip install optuna -q
# Graph / community detection
!pip install python-igraph leidenalg -q
# Neo4j (optional)
!pip install neo4j -q
!python -m spacy download en_core_web_sm -q

## 1. Configuration & Helpers

All tuneable parameters in one place. Ollama connectivity is verified on import.

In [ ]:
from __future__ import annotations
import json, os, re, time, logging, requests
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from datasets import load_dataset
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# Canonical relation schema used by both bilinear RE and optional LLM checks.
RELATION_TYPES = [
    "ACQUIRED", "MERGED_WITH", "INVESTED_IN", "ANNOUNCED", "REPORTED",
    "EMPLOYED_BY", "LOCATED_IN", "PARTNERED_WITH", "REGULATED_BY",
    "ISSUED", "SUBSIDIARY_OF", "COMPETES_WITH", "HAS_NUMERIC_FACT",
]

# Shared stopword set — used by entity validation across all pipeline stages.
_FINANCIAL_STOPWORDS = {
    "the","a","an","of","in","on","at","by","for","to","is","are","was","were",
    "be","been","being","have","has","had","do","does","did","will","would",
    "could","should","may","might","shall","what","which","who","whom","whose",
    "when","where","why","how","this","that","these","those","it","its","i",
    "he","she","we","they","you","me","him","her","us","them","my","your",
    "his","our","their","year","quarter","percent","period","time","day",
    "month","week","date","company","group","inc","corp","ltd","plc","co",
    "said","says","new","first","last","also","just","more","most","some",
    "between","after","before","during","over","under","into","from",
    "planned","outstanding","friend","list","s",
}

CFG = {
    # ── Ollama ─────────────────────────────────────────────────────────────
    "ollama_url"         : os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434"),
    "llm_model"          : "llama3.2:3b",
    "answer_model"       : "mistral-nemo",
    # ── Models ─────────────────────────────────────────────────────────────
    "plm_name"           : "ProsusAI/finbert",
    "embed_model"        : "sentence-transformers/all-MiniLM-L6-v2",
    # ── Corpus ─────────────────────────────────────────────────────────────
    "n_articles"         : 5000,
    "chunk_size"         : 512,
    "chunk_overlap"      : 64,
    # ── GCN ────────────────────────────────────────────────────────────────
    "gcn_hidden_dim"     : 256,
    "gcn_output_dim"     : 256,
    # ── Retrieval ──────────────────────────────────────────────────────────
    "semantic_threshold" : 0.7,
    "top_k_global"       : 5,
    "hop"                : 2,
    # ── SSHG weights (λ₁ syntactic, λ₂ semantic, λ₃ co-occurrence) ────────
    "sshg_lambda1"       : 1.0,    # syntactic dependency edge weight
    "sshg_lambda2"       : 0.8,    # cosine-similarity edge weight
    "sshg_lambda3"       : 0.5,    # co-occurrence edge weight
    # ── Supervised relation extraction ─────────────────────────────────────
    "rel_confidence_threshold" : 0.35,   # bilinear score threshold
    # ── Uncertainty / quality filtering ────────────────────────────────────
    "log_prob_threshold" : -2.5,   # min mean log-prob for accepted NER tokens
    "dep_threshold"      : 0.70,   # data-error-potential max (higher = noisier)
    # ── Temporal retrieval ─────────────────────────────────────────────────
    "time_window_days"   : 90,     # retrieval window around query date
    # ── Neo4j (optional) ───────────────────────────────────────────────────
    "neo4j_uri"          : os.environ.get("NEO4J_URI",      "bolt://localhost:7687"),
    "neo4j_user"         : os.environ.get("NEO4J_USER",     "neo4j"),
    "neo4j_password"     : os.environ.get("NEO4J_PASSWORD", "password"),
    # ── I/O ────────────────────────────────────────────────────────────────
    "qa_set_path"        : "qa_set.json",
    "results_path"       : "graphrag_results.json",
    "temperature"        : 0.0,
    # ── KG build ───────────────────────────────────────────────────────────
    "process_n_chunks"   : 500,
    "max_communities"    : 30,
    # ── HPO ────────────────────────────────────────────────────────────────
    "hpo_n_trials"       : 20,
    # ── Supervised NER fine-tuning ─────────────────────────────────────────
    "ner_dataset"        : "nlpaueb/finer-139",  # HF dataset
    "ner_finetune_epochs": 3,
    "ner_batch_size"     : 16,
    "ner_lr"             : 2e-5,
    "ner_max_seq_len"    : 128,
    "ner_model_dir"      : "finbert_ner_finetuned",        # local save path
    # ── Supervised relation fine-tuning ────────────────────────────────────
    "re_train_chunks"    : 200,
    "re_train_epochs"    : 3,
    "re_train_lr"        : 1e-3,
    "re_batch_size"      : 64,
    # ── Safety guard ───────────────────────────────────────────────────────
    "safety_model"       : "meta-llama/LlamaGuard-7b",
    "safety_enabled"     : False,   # flip to True when GPU + model available
}

def ollama(model: str, prompt: str, timeout: int = 120) -> str:
    """Call Ollama /api/generate. Returns response text or '[ERROR: ...]'."""
    try:
        r = requests.post(
            f"{CFG['ollama_url']}/api/generate",
            json={"model": model, "prompt": prompt, "stream": False,
                  "options": {"temperature": CFG["temperature"]}},
            timeout=timeout,
        )
        r.raise_for_status()
        return r.json().get("response", "").strip()
    except Exception as e:
        return f"[ERROR: {e}]"

# Verify Ollama connectivity
try:
    _pong = requests.get(f"{CFG['ollama_url']}/api/tags", timeout=5)
    _models = [m["name"] for m in _pong.json().get("models", [])]
    print(f"Ollama OK — available models: {_models}")
except Exception as _e:
    print(f"Ollama not reachable ({_e}). LLM calls will return '[ERROR: ...]'.")

## 2. Dataset Loading — Bloomberg Financial News 120K

Stream the first `n_articles` articles. Ticker symbols are detected via regex and stored for later anchoring.

In [ ]:
TICKER_PAT = re.compile(r'\(([A-Z]{1,5}(?::[A-Z]{2})?)\)|\$([A-Z]{1,5})\b')

print(f"Streaming {CFG['n_articles']:,} articles from Bloomberg Financial News 120K...")
_stream = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train", streaming=True)

articles: List[Dict] = []
for _i, _row in enumerate(tqdm(_stream, total=CFG["n_articles"], desc="Loading")):
    if _i >= CFG["n_articles"]:
        break
    _text = _row.get("Article") or _row.get("article") or ""
    if not isinstance(_text, str) or len(_text.strip()) < 30:
        continue
    _tickers = list({m.group(1) or m.group(2)
                     for m in TICKER_PAT.finditer(_text) if m.group(1) or m.group(2)})
    articles.append({
        "article_id" : _i,
        "headline"   : str(_row.get("Headline", "")),
        "text"       : re.sub(r'\s+', ' ', _text).strip(),
        "tickers"    : _tickers,
        "pub_date"   : str(_row.get("Date", "")) or None,
    })

print(f"Loaded {len(articles):,} articles")
print(f"Sample: [{articles[0]['article_id']}] {articles[0]['headline']} | tickers={articles[0]['tickers']}")

## 3. Preprocessing — Chunking

Split each article into overlapping fixed-size character windows.
- `chunk_size = 512` chars ≈ 2–3 sentences — enough for a full entity-relation triple
- `chunk_overlap = 64` chars — prevents relations being split at chunk boundaries

In [ ]:
def chunk_text(text: str, size: int = CFG["chunk_size"],
               overlap: int = CFG["chunk_overlap"]) -> List[str]:
    """Fixed-size character chunking with overlap."""
    chunks, i = [], 0
    while i < len(text):
        c = text[i : i + size].strip()
        if len(c) > 20:
            chunks.append(c)
        i += max(1, size - overlap)
    return chunks

article_chunks: List[Dict] = []
for art in articles:
    for j, chunk in enumerate(chunk_text(art["text"])):
        article_chunks.append({
            "chunk_id"   : f"{art['article_id']}_{j}",
            "article_id" : art["article_id"],
            "headline"   : art["headline"],
            "text"       : chunk,
            "pub_date"   : art["pub_date"],
            "tickers"    : art["tickers"],
        })

print(f"Total chunks : {len(article_chunks):,}")
print(f"Avg / article: {len(article_chunks)/len(articles):.1f}")

## 4. Named Entity Extraction — FinBERT + BIO Head

`ProsusAI/finbert` is pre-trained on financial corpora (Reuters, Bloomberg, earnings calls).
A linear BIO classification head is placed on top of the last hidden state.
Token-level BIO predictions are decoded into entity spans; each span's embedding
is obtained by mean-pooling its constituent token embeddings from FinBERT.

In [ ]:
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
)
from seqeval.metrics import classification_report as seq_report

# ── Label schema ──────────────────────────────────────────────────────────────
ENTITY_TYPES = ["ORG", "PER", "MONEY", "DATE", "GPE", "PRODUCT", "EVENT", "NORP"]
BIO_LABELS   = ["O"] + [f"B-{t}" for t in ENTITY_TYPES] + [f"I-{t}" for t in ENTITY_TYPES]
LABEL2ID     = {l: i for i, l in enumerate(BIO_LABELS)}
ID2LABEL     = {i: l for l, i in LABEL2ID.items()}

_SPACY_TYPE_MAP = {
    "ORG":"ORG","PERSON":"PER","MONEY":"MONEY","DATE":"DATE",
    "GPE":"GPE","LOC":"GPE","PRODUCT":"PRODUCT","EVENT":"EVENT",
    "NORP":"NORP","FAC":"ORG",
}

_ENTITY_STOPWORDS = _FINANCIAL_STOPWORDS

def _clean_entity_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip(" \t\n\r\"'`()[]{}.,:;!?-")

def is_valid_entity_name(name: str, entity_type: Optional[str] = None) -> bool:
    n = _clean_entity_text(name)
    if len(n) < 3: return False
    if n.lower() in _ENTITY_STOPWORDS: return False
    if not re.search(r"[A-Za-z]", n): return False
    if re.fullmatch(r"[\W_]+", n): return False
    if entity_type not in {"MONEY","DATE"} and n.lower()==n and len(n.split())==1:
        return False
    if entity_type=="MONEY" and not re.search(r"\d|\$|%|billion|million", n, re.I):
        return False
    return True

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Load tokenizer & base FinBERT ─────────────────────────────────────────────
ner_tokenizer = AutoTokenizer.from_pretrained(CFG["plm_name"])
print(f"Tokenizer loaded: {CFG['plm_name']}")

# ─────────────────────────────────────────────────────────────────────────────
# Fine-tuning helper — enforced path for NER model provisioning
# ─────────────────────────────────────────────────────────────────────────────
def _align_labels_with_tokens(labels, word_ids):
    """Propagate word-level BIO labels to subword tokens (first subword gets label,
    continuation subwords get I- if head was B-/I-, else -100 to ignore)."""
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)
        elif word_id != current_word:
            current_word = word_id
            new_labels.append(labels[word_id] if word_id < len(labels) else -100)
        else:
            # continuation subword: keep I- tag or mask
            lbl = labels[word_id] if word_id < len(labels) else -100
            if lbl != -100:
                label_str = ID2LABEL.get(lbl, "O")
                if label_str.startswith("B-"):
                    lbl = LABEL2ID.get("I-" + label_str[2:], lbl)
            new_labels.append(lbl)
    return new_labels

def _tokenize_and_align(examples, tokenizer, max_len=128):
    tokenized = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True,
        max_length=max_len, padding="max_length",
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        all_labels.append(_align_labels_with_tokens(labels, word_ids))
    tokenized["labels"] = all_labels
    return tokenized

def _compute_ner_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        tl, tp = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                tl.append(ID2LABEL[l])
                tp.append(ID2LABEL[p])
        true_labels.append(tl)
        true_preds.append(tp)
    report = seq_report(true_labels, true_preds, output_dict=True, zero_division=0)
    return {"f1": report.get("weighted avg", {}).get("f1-score", 0.0),
            "precision": report.get("weighted avg", {}).get("precision", 0.0),
            "recall": report.get("weighted avg", {}).get("recall", 0.0)}

def finetune_finbert_ner(
    dataset_name: str = CFG["ner_dataset"],
    save_dir: str = CFG["ner_model_dir"],
    force_train: bool = False,
) -> "AutoModelForTokenClassification":
    """Provision a trained FinBERT NER model by loading an existing checkpoint or training one."""
    if os.path.isdir(save_dir) and not force_train:
        print(f"Loading existing fine-tuned NER model from {save_dir} ...")
        return AutoModelForTokenClassification.from_pretrained(save_dir)

    print(f"Training NER model from dataset: {dataset_name} ...")
    raw = load_dataset(dataset_name, split='train[:10%]')

    ds_labels = raw["train"].features["ner_tags"].feature.names
    print(f"  Dataset labels ({len(ds_labels)}): {ds_labels[:10]} ...")

    _ds2ours = {}
    for ds_id, ds_lbl in enumerate(ds_labels):
        if ds_lbl == "O":
            _ds2ours[ds_id] = LABEL2ID["O"]
            continue
        prefix, etype = ds_lbl[:2], ds_lbl[2:].upper()
        mapped_type = _SPACY_TYPE_MAP.get(etype, etype)
        our_lbl = prefix + mapped_type
        _ds2ours[ds_id] = LABEL2ID.get(our_lbl, LABEL2ID["O"])

    def remap_tags(examples):
        examples["ner_tags"] = [[_ds2ours.get(t, 0) for t in seq]
                                 for seq in examples["ner_tags"]]
        return examples

    raw = raw.map(remap_tags, batched=True)

    tok_ds = raw.map(
        lambda ex: _tokenize_and_align(ex, ner_tokenizer, CFG["ner_max_seq_len"]),
        batched=True, remove_columns=raw["train"].column_names,
    )

    model = AutoModelForTokenClassification.from_pretrained(
        CFG["plm_name"],
        num_labels=len(BIO_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    train_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=CFG["ner_finetune_epochs"],
        per_device_train_batch_size=CFG["ner_batch_size"],
        per_device_eval_batch_size=CFG["ner_batch_size"],
        learning_rate=CFG["ner_lr"],
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    collator = DataCollatorForTokenClassification(ner_tokenizer)
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=tok_ds["train"],
        eval_dataset=tok_ds.get("validation", tok_ds["train"].select(range(min(500, len(tok_ds["train"]))))),
        processing_class=ner_tokenizer,
        data_collator=collator,
        compute_metrics=_compute_ner_metrics,
    )

    print("Fine-tuning FinBERT NER head ...")
    trainer.train()
    trainer.save_model(save_dir)
    ner_tokenizer.save_pretrained(save_dir)
    print(f"  Saved fine-tuned NER model → {save_dir}")
    return AutoModelForTokenClassification.from_pretrained(save_dir)

# Enforced path: always provision NER through the helper.
finbert_ner = finetune_finbert_ner()
_ner_tok = AutoTokenizer.from_pretrained(CFG["ner_model_dir"])

finbert_ner = finbert_ner.to(device).eval()
finbert     = finbert_ner.base_model   # shared FinBERT backbone for embeddings
print(f"NER model ready  ({sum(p.numel() for p in finbert_ner.parameters()):,} params)")

# ─────────────────────────────────────────────────────────────────────────────
# EntityMention dataclass (used throughout the notebook)
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class EntityMention:
    text        : str
    entity_type : str
    start       : int
    end         : int
    embedding   : Optional[np.ndarray] = None
    log_prob    : float = 0.0      # mean token log-probability (uncertainty signal)

# ─────────────────────────────────────────────────────────────────────────────
# NER inference — returns (entities, embeddings) with log-prob uncertainty
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def extract_entities_bio(
    text: str,
    log_prob_threshold: float = CFG["log_prob_threshold"],
) -> Tuple[List[EntityMention], np.ndarray]:
    """Run supervised FinBERT NER + collect per-entity mean log-probability.
    Entities whose mean log-prob < log_prob_threshold are dropped (uncertain)."""
    enc = _ner_tok(
        text[:512], return_tensors="pt", truncation=True,
        max_length=CFG["ner_max_seq_len"], return_offsets_mapping=True,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}

    outputs = finbert_ner(**enc, output_hidden_states=True)
    logits   = outputs.logits[0]                      # (T, num_labels)
    log_probs = torch.log_softmax(logits, dim=-1)     # (T, num_labels)
    pred_ids  = logits.argmax(dim=-1).tolist()

    # CLS embedding (for downstream GCN input)
    cls_emb = outputs.hidden_states[-1][0, 0].cpu().numpy()  # (hidden,)

    # Collect token-level predictions
    tokens_pred = []
    for t_i, (pred_id, (char_s, char_e)) in enumerate(zip(pred_ids, offsets)):
        if char_s == char_e == 0 and t_i > 0:
            continue                          # [SEP] / padding
        label  = ID2LABEL.get(pred_id, "O")
        lp     = log_probs[t_i, pred_id].item()
        tokens_pred.append((label, char_s, char_e, lp))

    # Decode BIO spans
    mentions: List[EntityMention] = []
    i = 0
    while i < len(tokens_pred):
        label, cs, ce, lp = tokens_pred[i]
        if label.startswith("B-"):
            etype = label[2:]
            span_s, span_e, log_ps = cs, ce, [lp]
            j = i + 1
            while j < len(tokens_pred) and tokens_pred[j][0] == f"I-{etype}":
                span_e = tokens_pred[j][2]
                log_ps.append(tokens_pred[j][3])
                j += 1
            mean_lp = float(np.mean(log_ps))
            if mean_lp >= log_prob_threshold:          # uncertainty filter
                span_text = text[span_s:span_e].strip()
                if is_valid_entity_name(span_text, etype):
                    mentions.append(EntityMention(
                        text=span_text, entity_type=etype,
                        start=span_s, end=span_e,
                        log_prob=mean_lp,
                    ))
            i = j
        else:
            i += 1

    # ── FinBERT embeddings for each span ─────────────────────────────────
    # Re-use cls_emb as a base; for each entity, extract mean hidden state
    # over its tokens from the last hidden layer
    hidden_last = outputs.hidden_states[-1][0]        # (T, hidden)
    embs = []
    for ment in mentions:
        tok_mask = [
            (s <= ment.start < e or ment.start <= s < ment.end)
            for (_, s, e, _) in tokens_pred
        ]
        idxs = [k for k, m in enumerate(tok_mask) if m]
        if idxs:
            span_emb = hidden_last[idxs].mean(0).cpu().numpy()
        else:
            span_emb = cls_emb
        ment.embedding = span_emb
        embs.append(span_emb)

    emb_arr = np.array(embs, dtype=np.float32) if embs else np.zeros((0, hidden_last.shape[-1]))
    return mentions, emb_arr

# Sanity check
_ents, _embs = extract_entities_bio(
    "Goldman Sachs announced a $2.5 billion acquisition of NN Investment Partners."
)
print(f"NER sanity check: {len(_ents)} entities")
for e in _ents:
    print(f"  [{e.entity_type:7s}  lp={e.log_prob:.2f}] '{e.text}'")

## 4b. Supervised Bilinear Relation Extractor — FinRE

A lightweight **bilinear scorer** sits on top of the FinBERT encoder:

$$s(h, t, r) = \mathbf{e}_h^\top \mathbf{W}_r \mathbf{e}_t$$

- Head / tail entity representations are **mean-pooled span embeddings** from FinBERT's last hidden layer.
- One weight matrix per relation type; training uses cross-entropy on FinRE-style triples.
- At inference, the argmax relation with score ≥ `rel_confidence_threshold` is accepted; below threshold the pair is discarded.
- Log-softmax scores double as **uncertainty signals** for downstream filtering.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

N_RELATIONS = len(RELATION_TYPES)

# ─────────────────────────────────────────────────────────────────────────────
# Bilinear Relation Classifier
# ─────────────────────────────────────────────────────────────────────────────
class BilinearRelationClassifier(nn.Module):
    """
    Bilinear scoring: s(h,t,r) = e_h' W_r e_t
    One weight matrix W_r per relation type (packed as a 3-D tensor).
    """
    def __init__(self, emb_dim: int, n_relations: int, dropout: float = 0.1):
        super().__init__()
        self.W       = nn.Parameter(torch.empty(n_relations, emb_dim, emb_dim))
        self.bias    = nn.Parameter(torch.zeros(n_relations))
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W)

    def forward(
        self,
        head_emb: torch.Tensor,   # (B, emb_dim)
        tail_emb: torch.Tensor,   # (B, emb_dim)
    ) -> torch.Tensor:            # (B, n_relations)
        h = self.dropout(head_emb)
        t = self.dropout(tail_emb)
        h_exp = h.unsqueeze(1)                           # (B, 1, emb)
        Wt    = torch.einsum("red,bd->bre", self.W, t) # (B, n_rel, emb)
        scores = (h_exp * Wt).sum(-1) + self.bias       # (B, n_rel)
        return scores

# ── Instantiate (emb_dim = FinBERT hidden size) ────────────────────────────
_emb_dim = finbert_ner.config.hidden_size
bilinear_re = BilinearRelationClassifier(_emb_dim, N_RELATIONS).to(device)
bilinear_re.eval()
print(f"BilinearRelationClassifier: {_emb_dim}→{N_RELATIONS} relations "
      f"| {sum(p.numel() for p in bilinear_re.parameters()):,} params")

# ─────────────────────────────────────────────────────────────────────────────
# Silver supervision for mandatory bilinear fine-tuning
# ─────────────────────────────────────────────────────────────────────────────
_RELATION_HINTS = {
    "ACQUIRED": ["acquired", "acquisition", "buyout", "purchased"],
    "MERGED_WITH": ["merged with", "merger"],
    "INVESTED_IN": ["invested in", "investment in", "stake in"],
    "ANNOUNCED": ["announced", "said", "declared"],
    "REPORTED": ["reported", "posted", "recorded"],
    "PARTNERED_WITH": ["partnered with", "partnership", "collaborat"],
    "ISSUED": ["issued", "sold", "offered"],
}

def _infer_relation_label(text: str) -> Optional[str]:
    t = text.lower()
    for rel, hints in _RELATION_HINTS.items():
        if any(h in t for h in hints):
            return rel
    return None

@torch.no_grad()
def build_silver_re_triples(chunks: List[Dict], max_chunks: int = CFG["re_train_chunks"]) -> List[Dict]:
    """Create lightweight silver triples to ensure supervised bilinear RE is trained before use."""
    triples: List[Dict] = []
    for chunk in tqdm(chunks[:max_chunks], desc="Silver RE triples"):
        rel = _infer_relation_label(chunk["text"])
        if rel is None:
            continue
        ents, _ = extract_entities_bio(chunk["text"])
        ents = [e for e in ents if e.embedding is not None and is_valid_entity_name(e.text, e.entity_type)]
        if len(ents) < 2:
            continue

        orgs = [e for e in ents if e.entity_type == "ORG"]
        pair_source = orgs if len(orgs) >= 2 else ents
        head, tail = pair_source[0], pair_source[1]
        if head.text == tail.text:
            continue

        triples.append({
            "head_emb": head.embedding[:_emb_dim],
            "tail_emb": tail.embedding[:_emb_dim],
            "relation_id": RELATION_TYPES.index(rel),
        })

    if not triples:
        raise RuntimeError(
            "No silver relation triples were generated. Increase CFG['re_train_chunks'] or relax hint rules."
        )
    return triples

# ─────────────────────────────────────────────────────────────────────────────
# Training helper — enforced usage in the main pipeline
# ─────────────────────────────────────────────────────────────────────────────
def finetune_bilinear_re(
    triples: List[Dict],
    epochs: int = CFG["re_train_epochs"],
    lr: float = CFG["re_train_lr"],
    batch_size: int = CFG["re_batch_size"],
) -> None:
    """Train the bilinear RE model on (head_emb, tail_emb, relation_id) triples."""
    if not triples:
        raise ValueError("finetune_bilinear_re requires at least one training triple.")

    bilinear_re.train()
    opt = torch.optim.AdamW(bilinear_re.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for ep in range(1, epochs + 1):
        random.shuffle(triples)
        total_loss, n_correct, n_total = 0.0, 0, 0
        for i in range(0, len(triples), batch_size):
            batch = triples[i : i + batch_size]
            h  = torch.tensor(np.stack([t["head_emb"] for t in batch]),
                               dtype=torch.float32, device=device)
            ta = torch.tensor(np.stack([t["tail_emb"] for t in batch]),
                               dtype=torch.float32, device=device)
            y  = torch.tensor([t["relation_id"] for t in batch],
                               dtype=torch.long, device=device)
            opt.zero_grad()
            scores = bilinear_re(h, ta)
            loss   = F.cross_entropy(scores, y)
            loss.backward()
            nn.utils.clip_grad_norm_(bilinear_re.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * len(batch)
            n_correct  += (scores.argmax(-1) == y).sum().item()
            n_total    += len(batch)
        sched.step()
        acc = n_correct / n_total if n_total else 0.0
        print(f"  RE epoch {ep}/{epochs} — loss={total_loss/n_total:.4f}  acc={acc:.3f}")
    bilinear_re.eval()

# Enforced path: train bilinear RE before inference.
silver_triples = build_silver_re_triples(article_chunks)
print(f"Silver triples: {len(silver_triples):,}")
finetune_bilinear_re(silver_triples)

# ─────────────────────────────────────────────────────────────────────────────
# Inference — supervised extraction path
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def extract_relations_bilinear(
    entities: List[EntityMention],
    provenance: Optional["Provenance"] = None,
    threshold: float = CFG["rel_confidence_threshold"],
) -> List["Relation"]:
    """Score all entity pairs with the bilinear model.
    Returns Relation objects for pairs whose top-class confidence >= threshold."""
    valid = [e for e in entities if e.embedding is not None
             and is_valid_entity_name(e.text, e.entity_type)]
    if len(valid) < 2:
        return []

    pairs, heads, tails = [], [], []
    for i in range(len(valid)):
        for j in range(len(valid)):
            if i == j:
                continue
            pairs.append((valid[i], valid[j]))
            heads.append(valid[i].embedding[:_emb_dim])
            tails.append(valid[j].embedding[:_emb_dim])

    H = torch.tensor(np.stack(heads), dtype=torch.float32, device=device)
    T = torch.tensor(np.stack(tails), dtype=torch.float32, device=device)
    scores    = bilinear_re(H, T)                        # (P, n_rel)
    log_probs = F.log_softmax(scores, dim=-1)
    best_lp, best_rid = log_probs.max(-1)                # (P,)

    rels = []
    for (ei, ej), lp, rid in zip(pairs, best_lp.tolist(), best_rid.tolist()):
        conf = float(torch.exp(torch.tensor(lp)).item())
        if conf < threshold:
            continue
        rtype = RELATION_TYPES[rid]
        rels.append(Relation(
            head=ei.text, head_type=ei.entity_type,
            relation_type=rtype,
            tail=ej.text, tail_type=ej.entity_type,
            confidence=conf,
            provenance=provenance,
            source="bilinear",
        ))
    return rels

print("Bilinear relation extractor ready (trained).")

## 5. SSHG — Syntax-Semantics Hybrid Graph

The SSHG operates at the **entity-mention level** (not raw tokens).

| Edge type | How added |
|-----------|-----------|
| **Syntactic** | Two entity mentions whose head tokens are connected by a dependency arc (`nsubj`, `dobj`, `prep`, `appos`, …) |
| **Semantic — co-occurrence** | Two entity mentions appear in the same sentence |
| **Semantic — cosine similarity** | Cosine similarity of FinBERT pooled embeddings ≥ threshold (0.7) |

In [ ]:
import networkx as nx
import spacy

print("Loading spaCy pipelines ...")
nlp_ner = spacy.load("en_core_web_sm")
nlp_dep = spacy.load("en_core_web_sm", disable=["ner"])
print("spaCy pipelines loaded.")

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-9 and nb > 1e-9 else 0.0

SYN_DEPS = {"nsubj", "dobj", "pobj", "prep", "appos", "compound", "nsubjpass", "csubj", "attr"}

def build_sshg(
    text: str,
    entities: List[EntityMention],
    threshold: float = CFG["semantic_threshold"],
    lambda1: float   = CFG["sshg_lambda1"],   # syntactic edge weight
    lambda2: float   = CFG["sshg_lambda2"],   # cosine-similarity edge weight
    lambda3: float   = CFG["sshg_lambda3"],   # co-occurrence edge weight
) -> nx.DiGraph:
    """Build Syntax-Semantics Hybrid Graph.

    Edge types and their configurable weights:
      - syntactic    (λ₁): dependency-arc edges between entity head tokens
      - semantic_sim (λ₂): cosine-similarity edges (weight = λ₂ × sim)
      - cooccurrence (λ₃): same-sentence co-occurrence edges
    """
    G = nx.DiGraph()
    if not entities:
        return G

    for i, ent in enumerate(entities):
        G.add_node(i, text=ent.text, type=ent.entity_type,
                   start=ent.start, end=ent.end)

    doc = nlp_dep(text[:1000])

    def head_tok(ent: EntityMention) -> Optional[int]:
        cands = [t for t in doc if t.idx >= ent.start and t.idx < ent.end]
        return cands[0].head.i if cands else None

    heads = [head_tok(e) for e in entities]
    sents = [(s.start_char, s.end_char) for s in doc.sents]

    for i, (ei, hi) in enumerate(zip(entities, heads)):
        for j, (ej, hj) in enumerate(zip(entities, heads)):
            if i >= j:
                continue

            # ── Syntactic edge (λ₁) ──────────────────────────────────────
            if hi is not None and hj is not None:
                ti, tj = doc[hi], doc[hj]
                if ti.head.i == hj and ti.dep_ in SYN_DEPS:
                    G.add_edge(i, j, edge_type="syntactic", dep=ti.dep_, weight=lambda1)
                elif tj.head.i == hi and tj.dep_ in SYN_DEPS:
                    G.add_edge(j, i, edge_type="syntactic", dep=tj.dep_, weight=lambda1)

            # ── Co-occurrence edge (λ₃) ──────────────────────────────────
            same = any(s[0] <= ei.start < s[1] and s[0] <= ej.start < s[1] for s in sents)
            if same and not G.has_edge(i, j):
                G.add_edge(i, j, edge_type="cooccurrence", weight=lambda3)
                G.add_edge(j, i, edge_type="cooccurrence", weight=lambda3)

            # ── Semantic similarity edge (λ₂) ────────────────────────────
            if ei.embedding is not None and ej.embedding is not None:
                sim = cosine_sim(ei.embedding, ej.embedding)
                if sim >= threshold and not G.has_edge(i, j):
                    G.add_edge(i, j, edge_type="semantic_sim", weight=float(sim) * lambda2)
                    G.add_edge(j, i, edge_type="semantic_sim", weight=float(sim) * lambda2)

    return G

# Sanity check
_ents, _embs = extract_entities_bio(
    "Goldman Sachs acquired NN Investment Partners for $1.6 billion in Q4 2021."
)
_sshg = build_sshg(
    "Goldman Sachs acquired NN Investment Partners for $1.6 billion in Q4 2021.",
    _ents,
)
_etypes = set(nx.get_edge_attributes(_sshg, "edge_type").values())
print(f"NER check : {len(_ents)} entities | SSHG: {_sshg.number_of_nodes()} nodes, "
      f"{_sshg.number_of_edges()} edges | edge types={_etypes}")
print(f"SSHG weights: λ₁={CFG['sshg_lambda1']}  λ₂={CFG['sshg_lambda2']}  λ₃={CFG['sshg_lambda3']}")

## 5b. Hyperparameter Optimisation — Optuna

Optuna searches over:

| Parameter | Range | Description |
|-----------|-------|-------------|
| `sshg_lambda1` | [0.3, 2.0] | Syntactic edge weight |
| `sshg_lambda2` | [0.1, 1.5] | Semantic similarity edge weight |
| `sshg_lambda3` | [0.1, 1.0] | Co-occurrence edge weight |
| `rel_confidence_threshold` | [0.20, 0.65] | Bilinear score cut-off |
| `semantic_threshold` | [0.50, 0.90] | Cosine similarity gate for semantic edges |

**Objective**: maximise **graph density × mean edge weight** on a 50-chunk calibration
subset (proxy for KG richness without a labelled gold graph).

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Calibration subset (50 chunks for speed) ──────────────────────────────
_HPO_N = min(50, len(article_chunks) if "article_chunks" in dir() else 0)

def _hpo_objective(trial: optuna.Trial) -> float:
    """Proxy objective: graph density × mean edge weight on _HPO_N chunks."""
    l1  = trial.suggest_float("sshg_lambda1",           0.3,  2.0)
    l2  = trial.suggest_float("sshg_lambda2",           0.1,  1.5)
    l3  = trial.suggest_float("sshg_lambda3",           0.1,  1.0)
    thr = trial.suggest_float("rel_confidence_threshold", 0.20, 0.65)
    sem = trial.suggest_float("semantic_threshold",     0.50, 0.90)

    if _HPO_N == 0:
        return 0.0          # no data yet — will run properly after dataset load

    total_density, total_weight = 0.0, 0.0
    n_valid = 0
    for chunk in article_chunks[:_HPO_N]:
        ents, _ = extract_entities_bio(chunk["text"])
        if len(ents) < 2:
            continue
        G = build_sshg(chunk["text"], ents,
                       threshold=sem, lambda1=l1, lambda2=l2, lambda3=l3)
        n = G.number_of_nodes()
        if n < 2:
            continue
        max_edges = n * (n - 1)
        density   = G.number_of_edges() / max_edges if max_edges else 0.0
        weights   = [d["weight"] for _, _, d in G.edges(data=True) if "weight" in d]
        mean_w    = float(np.mean(weights)) if weights else 0.0
        # Penalise over-dense graphs (spurious edges) with a mild penalty
        penalty   = max(0.0, density - 0.6)
        total_density += density - penalty
        total_weight  += mean_w
        n_valid       += 1

    if n_valid == 0:
        return 0.0
    return (total_density / n_valid) * (total_weight / n_valid)


def run_hpo(n_trials: int = CFG["hpo_n_trials"]) -> Dict:
    """Run Optuna study and apply best params to CFG."""
    print(f"Running HPO ({n_trials} trials) ...")
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_hpo_objective, n_trials=n_trials, show_progress_bar=True)
    best = study.best_params
    # ── Apply best params to CFG ──────────────────────────────────────────
    CFG["sshg_lambda1"]            = best["sshg_lambda1"]
    CFG["sshg_lambda2"]            = best["sshg_lambda2"]
    CFG["sshg_lambda3"]            = best["sshg_lambda3"]
    CFG["rel_confidence_threshold"]= best["rel_confidence_threshold"]
    CFG["semantic_threshold"]      = best["semantic_threshold"]
    print("\nBest HPO params:")
    for k, v in best.items():
        print(f"  {k:35s} = {v:.4f}")
    print(f"  Best value (proxy objective): {study.best_value:.5f}")
    return best

# ── Run HPO only if calibration data is available ────────────────────────
# (re-run this cell after dataset loading / chunking to activate search)
if _HPO_N >= 10:
    run_hpo()
else:
    print(f"HPO deferred — need ≥10 chunks (have {_HPO_N}).  "
          "Re-run after loading the dataset.")
    print(f"Current CFG weights: λ₁={CFG['sshg_lambda1']}  "
          f"λ₂={CFG['sshg_lambda2']}  λ₃={CFG['sshg_lambda3']}  "
          f"rel_thr={CFG['rel_confidence_threshold']}")

## 6. GCN Propagation — 2-Layer Graph Convolutional Network

$$h_i^{(l+1)} = \sigma\!\left(\hat{A}\, H^{(l)}\, W^{(l)}\right)$$

where $\hat{A} = D^{-1/2}(A+I)D^{-1/2}$ is the symmetrically normalised adjacency with self-loops.
Node features are initialised from FinBERT pooled embeddings.

In [ ]:
class GCNLayer(nn.Module):
    """GCN layer: Â X W with LayerNorm + ReLU."""
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.act  = nn.ReLU()
    def forward(self, X: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(A_hat @ self.W(X)))

class TwoLayerGCN(nn.Module):
    def __init__(self, in_dim: int, hid: int, out_dim: int):
        super().__init__()
        self.l1 = GCNLayer(in_dim, hid)
        self.l2 = GCNLayer(hid, out_dim)
    def forward(self, X: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        return self.l2(self.l1(X, A_hat), A_hat)

def norm_adj(G: nx.DiGraph, n: int) -> torch.Tensor:
    """Symmetric normalised adjacency Â = D^{-½}(A+I)D^{-½}."""
    A = np.zeros((n, n), dtype=np.float32)
    for u, v, d in G.edges(data=True):
        if u < n and v < n:
            A[u, v] = d.get("weight", 1.0)
    A = (A + A.T) / 2 + np.eye(n, dtype=np.float32)
    D_inv_sqrt = np.diag(A.sum(1).clip(1e-9) ** -0.5)
    return torch.tensor(D_inv_sqrt @ A @ D_inv_sqrt, dtype=torch.float32)

gcn = TwoLayerGCN(finbert.config.hidden_size,
                  CFG["gcn_hidden_dim"],
                  CFG["gcn_output_dim"]).to(device)
gcn.eval()
print(f"GCN: {finbert.config.hidden_size} -> {CFG['gcn_hidden_dim']} -> {CFG['gcn_output_dim']}")

@torch.no_grad()
def apply_gcn(entities: List[EntityMention], sshg: nx.DiGraph) -> np.ndarray:
    """Apply 2-layer GCN; returns refined embeddings (N, gcn_output_dim)."""
    n = len(entities)
    if n == 0:
        return np.zeros((0, CFG["gcn_output_dim"]), dtype=np.float32)
    feat_dim = finbert.config.hidden_size
    X = np.zeros((n, feat_dim), dtype=np.float32)
    for i, ent in enumerate(entities):
        if ent.embedding is not None:
            e = ent.embedding[:feat_dim]
            X[i, :len(e)] = e
    Xt    = torch.tensor(X, dtype=torch.float32).to(device)
    A_hat = norm_adj(sshg, n).to(device)
    return gcn(Xt, A_hat).cpu().numpy()

_refined = apply_gcn(_ents, _sshg)
print(f"GCN output: {_refined.shape}")

## 7. Temporal Anchoring, Provenance & Numeric Facts

- **Dates** are extracted with regex + `dateutil` and normalised to ISO 8601.
  The document publication date is used as a fallback.
- **Provenance** attaches `chunk_id`, source sentence, `doc_id`, and `doc_timestamp` to every relation.
- **Numeric facts** (revenue, deal values, percentages) are stored as edge properties with
  `relation_type = "HAS_NUMERIC_FACT"` on self-loop edges.

In [ ]:
from dateutil import parser as dateparser

_DATE_PATS = [
    re.compile(r'\b(\d{4}-\d{2}-\d{2})\b'),
    re.compile(r'\b(Q[1-4]\s+\d{4})\b'),
    re.compile(r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
               r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|'
               r'Dec(?:ember)?)\s+\d{1,2},?\s+\d{4}\b', re.IGNORECASE),
    re.compile(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}\b',
               re.IGNORECASE),
    re.compile(r'\b(\d{4})\b'),
]
_NUM_PAT = re.compile(
    r'(\$\s*[\d,]+(?:\.\d+)?(?:\s*(?:billion|million|thousand|[BMK]))?|'
    r'[\d,]+(?:\.\d+)?\s*(?:percent|%|basis\s+points?|bps?))',
    re.IGNORECASE,
)

def extract_dates(text: str, fallback: Optional[str] = None) -> List[str]:
    seen, out = set(), []
    for pat in _DATE_PATS:
        for m in pat.finditer(text):
            raw = m.group(0).strip()
            if raw in seen: continue
            seen.add(raw)
            try:
                parsed = dateparser.parse(raw)
                out.append(parsed.strftime("%Y-%m-%d") if parsed else raw)
            except Exception:
                out.append(raw)
    return out or ([fallback] if fallback else [])

@dataclass
class Provenance:
    chunk_id      : str
    sentence      : str
    doc_id        : int
    doc_timestamp : Optional[str]
    def to_dict(self) -> Dict:
        return {"chunk_id": self.chunk_id, "sentence": self.sentence,
                "doc_id": self.doc_id, "doc_timestamp": self.doc_timestamp}

@dataclass
class NumericFact:
    metric        : str
    raw_value     : str
    unit          : str
    time_interval : Optional[str]
    def to_dict(self) -> Dict:
        return {"metric": self.metric, "raw_value": self.raw_value,
                "unit": self.unit, "time_interval": self.time_interval}

_METRIC_WORDS = ["revenue","earnings","profit","sales","deal","acquisition",
                 "value","loss","income","gain","dividend","assets"]

def extract_numeric_facts(text: str, dates: List[str]) -> List[NumericFact]:
    facts = []
    for m in _NUM_PAT.finditer(text):
        raw  = m.group(0).strip()
        unit = ("USD"     if "$" in raw
                else "percent"  if "%" in raw or "percent" in raw.lower()
                else "bps"      if "basis" in raw.lower()
                else "number")
        ctx    = text[max(0, m.start()-40): m.start()].lower()
        metric = next((w for w in _METRIC_WORDS if w in ctx), "value")
        facts.append(NumericFact(metric, raw, unit, dates[0] if dates else None))
    return facts

# Test
_dates = extract_dates(articles[0]["text"][:500], articles[0]["pub_date"])
_facts = extract_numeric_facts(articles[0]["text"][:500], _dates)
print(f"Dates : {_dates[:3]}")
print(f"Numeric facts: {len(_facts)}")
for f in _facts[:3]: print(f"  {f.metric}: {f.raw_value} ({f.unit})")

## 8. Relation Integration

This section defines the relation container and unified extraction entry point used by graph construction.

All relation inference is performed by the supervised bilinear extractor trained earlier.

In [ ]:
from rapidfuzz import fuzz

# ── Relation dataclass ────────────────────────────────────────────────────────
@dataclass
class Relation:
    head          : str
    head_type     : str
    relation_type : str
    tail          : str
    tail_type     : str
    confidence    : float
    provenance    : Optional[Provenance] = None
    time_interval : Optional[str]        = None
    numeric_facts : List[NumericFact]    = field(default_factory=list)
    source        : str                  = "bilinear"   # "bilinear" | "llm"

# ── LLM relation extraction (fallback / cross-validation) ────────────────────
_RE_PROMPT = """You are a financial information extraction expert.
Given the news text and the named entities found in it, extract ALL meaningful relationships.

Text: {text}
Known entities: {entities}

Output ONLY valid JSON objects, one per line — no markdown, no preamble:
{{\"head\": \"EntityA\", \"head_type\": \"ORG|PER|GPE|...\", \"relation\": \"RELATION_TYPE\", \"tail\": \"EntityB\", \"tail_type\": \"ORG|...\", \"confidence\": 0.85, \"evidence\": \"short quote\"}}

Valid relation types: {rel_types}
Include implicit and cross-sentence relations. Output JSON lines only:"""

def _match_entity_name(raw_name: str, entities: List[EntityMention]) -> Optional[Tuple[str, str]]:
    candidate = _clean_entity_text(raw_name)
    if not candidate: return None
    best, best_score = None, -1.0
    for e in entities:
        score = max(fuzz.ratio(candidate.lower(), e.text.lower()),
                    fuzz.partial_ratio(candidate.lower(), e.text.lower()))
        if score > best_score:
            best_score, best = score, e
    return (best.text, best.entity_type) if best and best_score >= 86 else None

def extract_relations_llm(
    chunk: Dict,
    entities: List[EntityMention],
    max_chars: int = 800,
) -> List[Relation]:
    """LLM-based fallback relation extraction (runs every rel_llm_fallback_every chunks)."""
    grounded = [e for e in entities if is_valid_entity_name(e.text, e.entity_type)]
    if len(grounded) < 2:
        return []
    ent_str = ", ".join(f"{e.text} ({e.entity_type})" for e in grounded[:15])
    raw = ollama(CFG["llm_model"],
                 _RE_PROMPT.format(text=chunk["text"][:max_chars],
                                   entities=ent_str,
                                   rel_types=", ".join(RELATION_TYPES)),
                 timeout=60)
    rels = []
    for line in raw.strip().split("\n"):
        try:
            m = re.search(r"\{.*\}", line, re.DOTALL)
            if not m: continue
            obj = json.loads(m.group())
            if not all(k in obj for k in ("head","relation","tail")): continue
            mh = _match_entity_name(obj.get("head",""), grounded)
            mt = _match_entity_name(obj.get("tail",""), grounded)
            if not mh or not mt: continue
            head_name, head_type = mh
            tail_name, tail_type = mt
            if head_name == tail_name: continue
            rtype = str(obj["relation"]).upper().replace(" ","_")
            if rtype not in RELATION_TYPES: continue
            confidence = float(obj.get("confidence", 0.0))
            if confidence < CFG["rel_confidence_threshold"]: continue
            prov = Provenance(chunk["chunk_id"],
                              str(obj.get("evidence",""))[:150],
                              chunk["article_id"], chunk["pub_date"])
            rels.append(Relation(head=head_name, head_type=head_type,
                                  relation_type=rtype,
                                  tail=tail_name, tail_type=tail_type,
                                  confidence=confidence, provenance=prov,
                                  source="llm"))
        except Exception:
            continue
    return rels

# ── Unified extraction: bilinear primary, LLM fallback ───────────────────────
def extract_relations(
    chunk: Dict,
    entities: List[EntityMention],
    chunk_idx: int = 0,
) -> List[Relation]:
    """
    Primary path  : supervised bilinear extractor (fast, deterministic).
    Fallback path : LLM every `rel_llm_fallback_every` chunks (slower, broader).
    Results from both paths are merged; bilinear wins on duplicates (same head/tail).
    """
    prov = Provenance(chunk["chunk_id"], chunk["text"][:150],
                      chunk["article_id"], chunk["pub_date"])

    # 1. Bilinear (always)
    bi_rels = extract_relations_bilinear(entities, provenance=prov,
                                         threshold=CFG["rel_confidence_threshold"])

    # 2. LLM fallback every N chunks
    llm_rels = []
    if chunk_idx % CFG["rel_llm_fallback_every"] == 0:
        llm_rels = extract_relations_llm(chunk, entities)

    # 3. Merge — dedup by (head, tail) keeping highest confidence
    seen: Dict[Tuple[str,str], Relation] = {}
    for r in bi_rels + llm_rels:
        key = (r.head.lower(), r.tail.lower(), r.relation_type)
        if key not in seen or r.confidence > seen[key].confidence:
            seen[key] = r

    return list(seen.values())

print(f"Relation extractor ready (bilinear primary + LLM fallback every "
      f"{CFG['rel_llm_fallback_every']} chunks | threshold={CFG['rel_confidence_threshold']}).")

## 9. Graph Construction — Neo4j (+ NetworkX Fallback)

An **append-only property graph** where:
- **Nodes** carry `canonical_id`, `name`, `type`, `first_seen`, `last_seen`
- **Edges** carry `relation_type`, `confidence`, `time_interval`, `provenance` (JSON), `numeric_facts`
- Parallel edges are allowed (same entity pair, different time or source)

Neo4j is used when available. All operations also maintain an in-memory `nx.MultiDiGraph`
so the rest of the pipeline works even without a running database.

In [ ]:
from collections import defaultdict

G_knowledge    : nx.MultiDiGraph = nx.MultiDiGraph()
entity_registry: Dict[str, Dict] = {}
community_store: Dict[int, Dict] = {}

# Try Neo4j
NEO4J_AVAILABLE = False
neo4j_driver    = None
try:
    from neo4j import GraphDatabase
    neo4j_driver = GraphDatabase.driver(
        CFG["neo4j_uri"], auth=(CFG["neo4j_user"], CFG["neo4j_password"])
    )
    neo4j_driver.verify_connectivity()
    NEO4J_AVAILABLE = True
    print(f"Neo4j connected: {CFG['neo4j_uri']}")
except Exception as _e:
    print(f"Neo4j not available ({_e}). Using in-memory NetworkX graph.")

def canonical_id(name: str, etype: str) -> str:
    return f"{etype.lower()}_{re.sub(r'[^a-z0-9]+', '_', name.strip().lower())}"

def upsert_entity(name: str, etype: str, ts: Optional[str] = None) -> Optional[str]:
    cleaned = _clean_entity_text(name)
    if not is_valid_entity_name(cleaned, etype):
        return None

    cid = canonical_id(cleaned, etype)
    if cid not in entity_registry:
        entity_registry[cid] = {
            "canonical_id": cid,
            "name": cleaned,
            "type": etype,
            "first_seen": ts,
            "last_seen": ts,
        }
        G_knowledge.add_node(cid, **entity_registry[cid])
    elif ts and (not entity_registry[cid]["last_seen"] or ts > entity_registry[cid]["last_seen"]):
        entity_registry[cid]["last_seen"] = ts
        if G_knowledge.has_node(cid):
            G_knowledge.nodes[cid]["last_seen"] = ts
    return cid

def add_relation(rel: Relation) -> None:
    hid = upsert_entity(rel.head, rel.head_type, rel.time_interval)
    tid = upsert_entity(rel.tail, rel.tail_type, rel.time_interval)
    if not hid or not tid or hid == tid:
        return

    props = {
        "relation_type": rel.relation_type,
        "confidence": rel.confidence,
        "time_interval": rel.time_interval or "",
        "source": rel.source,
        "provenance": json.dumps(rel.provenance.to_dict()) if rel.provenance else "{}",
        "numeric_facts": json.dumps([nf.to_dict() for nf in rel.numeric_facts]),
    }
    G_knowledge.add_edge(hid, tid, **props)

    if NEO4J_AVAILABLE:
        try:
            with neo4j_driver.session() as sess:
                sess.run(
                    """
                    MERGE (h:Entity {canonical_id:$hid}) SET h.name=$hn, h.type=$ht
                    MERGE (t:Entity {canonical_id:$tid}) SET t.name=$tn, t.type=$tt
                    CREATE (h)-[r:RELATION {relation_type:$rt, confidence:$cf,
                        time_interval:$ti, source:$src, provenance:$pv}]->(t)
                    """,
                    hid=hid, hn=entity_registry[hid]["name"], ht=rel.head_type,
                    tid=tid, tn=entity_registry[tid]["name"], tt=rel.tail_type,
                    rt=rel.relation_type, cf=rel.confidence,
                    ti=props["time_interval"], src=rel.source, pv=props["provenance"],
                )
        except Exception as neo_e:
            logger.warning("Neo4j write: %s", neo_e)

def add_numeric_edge(ent_name: str, etype: str, fact: NumericFact, prov: Provenance):
    # Keep numeric self-loops only on stronger anchor entity types to reduce noise.
    if etype not in {"ORG", "PER", "GPE", "PRODUCT"}:
        return
    if not fact or not fact.metric or fact.metric == "value":
        return

    cid = upsert_entity(ent_name, etype, fact.time_interval)
    if not cid:
        return

    G_knowledge.add_edge(
        cid, cid,
        relation_type="HAS_NUMERIC_FACT",
        metric=fact.metric, raw_value=fact.raw_value, unit=fact.unit,
        time_interval=fact.time_interval or "",
        provenance=json.dumps(prov.to_dict()),
        confidence=1.0, source="rule",
        numeric_facts=json.dumps([fact.to_dict()]),
    )

print("Graph builder ready.")

## 10. Knowledge Graph Construction — Full Pipeline per Chunk

For each chunk:
1. **FinBERT + BIO** → entity mentions + embeddings
2. **SSHG** → entity-level graph from dependency + co-occurrence + cosine similarity
3. **GCN** → refine entity embeddings via neighbourhood aggregation
4. **Temporal / Numeric** → extract dates and numeric facts
5. **Supervised RE** → relation extraction via trained bilinear classifier
6. **Graph write** → upsert entities + edges into `G_knowledge` (and Neo4j if available)

> Set `CFG["process_n_chunks"] = len(article_chunks)` to process the full 5 000-article corpus.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Data Error Potential (DEP) filter
# ─────────────────────────────────────────────────────────────────────────────
def data_error_potential(chunk: Dict, entities: List[EntityMention]) -> float:
    """Estimate the noisiness of a chunk (higher = noisier).

    Signals (each in [0,1]):
      1. Low mean NER log-probability  → model uncertain about extractions
      2. High entity density           → likely boilerplate / list text
      3. Very short chunks             → insufficient context
      4. Abnormally high entity count  → repeated tokens, header artefacts
    """
    if not entities:
        return 1.0       # no entities → skip

    mean_lp  = float(np.mean([e.log_prob for e in entities]))
    lp_score = max(0.0, min(1.0, (-mean_lp) / 5.0))    # map [-5,0] → [1,0]

    n_ents   = len(entities)
    n_chars  = max(len(chunk["text"]), 1)
    ent_density = min(1.0, n_ents / (n_chars / 30))    # expect ~1 ent / 30 chars

    short_penalty = 1.0 if n_chars < 80 else 0.0
    count_penalty = min(1.0, max(0.0, (n_ents - 20) / 20))  # > 20 entities is suspicious

    dep = 0.4 * lp_score + 0.3 * ent_density + 0.2 * short_penalty + 0.1 * count_penalty
    return float(dep)

# ─────────────────────────────────────────────────────────────────────────────
# Full KG construction pipeline — with DEP filter and uncertainty tracking
# ─────────────────────────────────────────────────────────────────────────────
PROCESS_N = min(CFG["process_n_chunks"], len(article_chunks))
print(f"Building KG from {PROCESS_N:,} / {len(article_chunks):,} chunks ...")
print(f"(DEP threshold={CFG['dep_threshold']}  log_prob_threshold={CFG['log_prob_threshold']})")

_errors  = 0
_skipped_dep = 0
_skipped_empty = 0

for _idx, chunk in enumerate(tqdm(article_chunks[:PROCESS_N], desc="KG build")):
    try:
        # 1. Supervised FinBERT NER + log-prob uncertainty filter
        ents, _ = extract_entities_bio(chunk["text"],
                                       log_prob_threshold=CFG["log_prob_threshold"])
        if not ents:
            _skipped_empty += 1
            continue

        # 2. Data Error Potential filter — drop noisy chunks
        dep_score = data_error_potential(chunk, ents)
        if dep_score > CFG["dep_threshold"]:
            _skipped_dep += 1
            continue

        # 3. SSHG with tuned λ weights
        sshg = build_sshg(chunk["text"], ents)

        # 4. GCN embedding refinement
        if sshg.number_of_nodes() > 1:
            refined = apply_gcn(ents, sshg)
            for _j, ent in enumerate(ents):
                if _j < len(refined):
                    ent.embedding = refined[_j]

        # 5. Temporal + numeric
        dates    = extract_dates(chunk["text"], chunk["pub_date"])
        num_facts= extract_numeric_facts(chunk["text"], dates)
        ts       = dates[0] if dates else chunk["pub_date"]

        prov = Provenance(chunk["chunk_id"], chunk["text"][:150],
                          chunk["article_id"], chunk["pub_date"])

        # 6. Supervised bilinear RE + LLM fallback
        all_rels = extract_relations(chunk, ents, chunk_idx=_idx)

        # 7. Write to graph
        for ent in ents:
            upsert_entity(ent.text, ent.entity_type, ts)

        for rel in all_rels:
            rel.time_interval = ts
            rel.numeric_facts = num_facts[:2]
            add_relation(rel)

        # Numeric self-loop edges for top entities
        for ent in ents[:3]:
            for nf in num_facts[:2]:
                add_numeric_edge(ent.text, ent.entity_type, nf, prov)

    except Exception as _e:
        _errors += 1
        logger.warning("Chunk %s error: %s", chunk["chunk_id"], _e)

print(f"\nKnowledge Graph built:")
print(f"  Entities (nodes)  : {G_knowledge.number_of_nodes():,}")
print(f"  Relations (edges) : {G_knowledge.number_of_edges():,}")
print(f"  Skipped (empty)   : {_skipped_empty:,}")
print(f"  Skipped (DEP)     : {_skipped_dep:,}")
print(f"  Errors            : {_errors:,}")

## 10b. Factuality / Hallucination Checks & Safety Monitoring

Two complementary quality gates run **after** KG construction and **before** answers are
returned to the user:

| Gate | Mechanism | When |
|------|-----------|------|
| **Factuality** | Checks every extracted relation against source text (context-grounding) via LLM | at relation write time (sampled) |
| **Hallucination filter** | Flags answers where claims are unsupported by retrieved context | at answer generation time |
| **Llama Guard** | Safety / bias scan on generated answers | at answer generation time (optional) |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Factuality check — is this relation grounded in the source text?
# ─────────────────────────────────────────────────────────────────────────────
_FACT_CHECK_PROMPT = """You are a factuality auditor for a financial knowledge graph.
Determine whether the extracted relation is directly supported by the source text.

Source text: {text}
Extracted relation: ({head}) -{relation}-> ({tail})

Answer with exactly one word: supported  OR  unsupported"""

def check_relation_factuality(rel: Relation, source_text: str) -> bool:
    """Return True if the relation is grounded in source_text (sampled check)."""
    raw = ollama(
        CFG["llm_model"],
        _FACT_CHECK_PROMPT.format(
            text=source_text[:600],
            head=rel.head, relation=rel.relation_type, tail=rel.tail,
        ),
        timeout=30,
    ).lower()
    return "supported" in raw

# ─────────────────────────────────────────────────────────────────────────────
# Hallucination filter — scan final answer for unsupported claims
# ─────────────────────────────────────────────────────────────────────────────
_HALLUC_PROMPT = """You are a hallucination detector for a financial QA system.
Check whether every factual claim in the answer is explicitly supported by the context.

Context:
{context}

Answer:
{answer}

Output ONLY valid JSON:
{{\"hallucination_detected\": true|false, \"unsupported_claims\": [\"...\"]}}"""

def detect_hallucination(answer: str, context: str) -> Dict:
    """Returns dict with hallucination_detected (bool) and unsupported_claims (list)."""
    raw = ollama(
        CFG["llm_model"],
        _HALLUC_PROMPT.format(context=context[:1200], answer=answer[:800]),
        timeout=45,
    )
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            result = json.loads(m.group())
            return {
                "hallucination_detected": bool(result.get("hallucination_detected", False)),
                "unsupported_claims":     result.get("unsupported_claims", []),
            }
    except Exception:
        pass
    return {"hallucination_detected": False, "unsupported_claims": []}

# ─────────────────────────────────────────────────────────────────────────────
# Llama Guard — safety / bias monitoring
# Runs via Ollama if the guard model is available; silently skips otherwise
# ─────────────────────────────────────────────────────────────────────────────
_GUARD_PROMPT = """[INST] Task: Check if there is unsafe content in the assistant message.
<BEGIN ASSISTANT MESSAGE>
{response}
<END ASSISTANT MESSAGE>

Provide your safety assessment for the assistant message above:
- First line must read \'safe\' or \'unsafe\'
- If unsafe, list the violated categories on a new line. [/INST]"""

def llama_guard_check(response: str) -> Dict:
    """"Run Llama Guard safety check. Returns {safe: bool, categories: list}."""
    if not CFG["safety_enabled"]:
        return {"safe": True, "categories": []}
    raw = ollama(CFG["safety_model"], _GUARD_PROMPT.format(response=response[:800]),
                 timeout=30)
    first_line = raw.strip().split("\n")[0].lower()
    categories = []
    if "unsafe" in first_line:
        lines = raw.strip().split("\n")
        categories = [l.strip() for l in lines[1:] if l.strip()]
    return {"safe": "unsafe" not in first_line, "categories": categories}

# ─────────────────────────────────────────────────────────────────────────────
# Sampled factuality audit on the freshly built KG
# (checks a random 5 % of relations that have provenance)
# ─────────────────────────────────────────────────────────────────────────────
import random as _rand

_audit_rels = []
for _u, _v, _d in G_knowledge.edges(data=True):
    prov_str = _d.get("provenance", "")
    if prov_str:
        try:
            prov_data = json.loads(prov_str) if isinstance(prov_str, str) else prov_str
        except Exception:
            prov_data = {}
        _audit_rels.append((_u, _v, _d, prov_data.get("sentence","")))

_sample_size = max(1, int(len(_audit_rels) * 0.05))
_sample = _rand.sample(_audit_rels, min(_sample_size, len(_audit_rels)))

_n_supported = 0
for _u, _v, _d, _sent in _sample:
    if not _sent:
        _n_supported += 1    # no provenance to check → give benefit of doubt
        continue
    _rel_obj = Relation(
        head=entity_registry.get(_u,{}).get("name","?"),
        head_type=entity_registry.get(_u,{}).get("type","?"),
        relation_type=_d.get("relation_type","REL"),
        tail=entity_registry.get(_v,{}).get("name","?"),
        tail_type=entity_registry.get(_v,{}).get("type","?"),
        confidence=_d.get("confidence",0.0),
    )
    if check_relation_factuality(_rel_obj, _sent):
        _n_supported += 1

_support_rate = _n_supported / len(_sample) if _sample else 0.0
print(f"Factuality audit: {_n_supported}/{len(_sample)} sampled relations "
      f"supported  ({_support_rate:.1%})")
print("Hallucination detector and Llama Guard safety monitor registered.")

## 11. Hierarchy & Communities — Leiden Algorithm

The Leiden algorithm partitions the entity graph into communities, which become
the global retrieval index. Each community receives an LLM-generated summary
grounded exclusively in intra-community relations.

In [ ]:
try:
    import igraph as ig
    import leidenalg
    _LEIDEN_OK = True
except ImportError:
    _LEIDEN_OK = False
    print("leidenalg/igraph not found — install: pip install leidenalg python-igraph")

_SUMMARY_PROMPT = """You are a financial knowledge analyst.
Summarise this cluster of related financial entities and their connections in 2-3 sentences.
Focus on the financial theme or event that links these entities.

Entities: {entities}
Relations: {relations}

Concise summary (2-3 sentences, facts only):"""

def run_leiden(G: nx.MultiDiGraph) -> Dict[str, int]:
    if not _LEIDEN_OK or G.number_of_nodes() == 0:
        return {}
    nodes     = list(G.nodes())
    n2i       = {n: i for i, n in enumerate(nodes)}
    undirected= G.to_undirected()
    edges_ig  = [(n2i[u], n2i[v]) for u, v in undirected.edges()
                 if u in n2i and v in n2i]
    g_ig      = ig.Graph(n=len(nodes), edges=edges_ig, directed=False)
    g_ig.simplify()
    part      = leidenalg.find_partition(g_ig, leidenalg.ModularityVertexPartition)
    return {nodes[i]: part.membership[i] for i in range(len(nodes))}

def build_community_summaries(G: nx.MultiDiGraph, membership: Dict[str, int],
                               cap: int = CFG["max_communities"]) -> Dict[int, Dict]:
    by_comm: Dict[int, List[str]] = defaultdict(list)
    for node, cid in membership.items():
        by_comm[cid].append(G.nodes[node].get("name", node))

    summaries: Dict[int, Dict] = {}
    for cid in tqdm(sorted(by_comm)[:cap], desc="Community summaries"):
        ents = by_comm[cid]
        comm_nodes = {n for n, c in membership.items() if c == cid}
        rels_txt = []
        for u, v, d in G.edges(data=True):
            if u in comm_nodes and v in comm_nodes:
                un = G.nodes[u].get("name", u)
                vn = G.nodes[v].get("name", v)
                rels_txt.append(f"{un} -{d.get('relation_type','REL')}-> {vn}")
        summary = ollama(
            CFG["llm_model"],
            _SUMMARY_PROMPT.format(entities=", ".join(ents[:10]),
                                   relations="; ".join(rels_txt[:10]) or "none"),
            timeout=45,
        )
        summaries[cid] = {"community_id": cid, "entities": ents,
                          "n_entities": len(ents), "summary": summary}
    return summaries

print("Running Leiden community detection ...")
if G_knowledge.number_of_nodes() > 0:
    membership = run_leiden(G_knowledge)
    n_comms = len(set(membership.values()))
    print(f"Communities found: {n_comms}")
    for node, cid in membership.items():
        if G_knowledge.has_node(node):
            G_knowledge.nodes[node]["community_id"] = cid
    print(f"Generating summaries (cap={CFG['max_communities']}) ...")
    community_store = build_community_summaries(G_knowledge, membership)
    print(f"Summaries generated: {len(community_store)}")
    if community_store:
        _sc = next(iter(community_store.values()))
        print(f"\nSample community ({_sc['n_entities']} entities):")
        print(f"  {', '.join(_sc['entities'][:5])}")
        print(f"  {_sc['summary'][:200]}")
else:
    membership = {}
    print("Graph empty — skipping Leiden.")
    
community_membership = membership

---
# Part 2 — Knowledge Retrieval & QA (Inference)

Given a user question, the pipeline:
1. Classifies the query as `local` / `global` / `mixed`
2. Routes to the appropriate retriever
3. Generates an answer with **Mistral-NeMo** using only the retrieved context
4. Evaluates with the **RAG Triad** (Faithfulness, Answer Relevance, Context Precision)

## 12. Query Type Determination

In [ ]:
_QT_PROMPT = """Classify this financial question into exactly one type:
- local  : asks about a specific entity (company, person, place)
- global : asks about broad themes, trends, or sector-wide patterns
- mixed  : compares entities or combines specific + thematic reasoning

Question: {question}

Output ONE word only (local / global / mixed):"""

def classify_query(question: str) -> str:
    raw = ollama(CFG["llm_model"], _QT_PROMPT.format(question=question)).lower()
    for qt in ("local", "global", "mixed"):
        if qt in raw:
            return qt
    return "local"

# Smoke test
for _q, _expect in [
    ("What did Goldman Sachs acquire in 2022?", "local"),
    ("What were major acquisition trends in 2022?", "global"),
]:
    _got = classify_query(_q)
    print(f"  '{_q[:55]}...' -> {_got}  (expected {_expect})")

## 13. Query Router — Local / Global / Hybrid Retrieval

| Route | Mechanism |
|-------|-----------|
| **Local** | Extract entity names from query → map to canonical IDs → BFS 2-hop subgraph → serialise as text |
| **Global** | Embed query with `all-MiniLM-L6-v2` → cosine similarity vs community summary embeddings → top-k |
| **Hybrid** | Concatenate local subgraph + global community summaries |

In [ ]:
from sentence_transformers import SentenceTransformer
from datetime import datetime, timedelta

print(f"Loading embedding model: {CFG['embed_model']} ...")
embedder = SentenceTransformer(CFG["embed_model"], device="cpu")
print("Embedding model ready.")

# Pre-compute community summary embeddings
_comm_list = list(community_store.values())
if _comm_list:
    _comm_texts = [c["summary"] for c in _comm_list]
    _comm_embs  = embedder.encode(_comm_texts, normalize_embeddings=True,
                                   show_progress_bar=False).astype(np.float32)
    # Stash timestamps for temporal filtering
    _comm_dates = [c.get("earliest_date") or c.get("latest_date") for c in _comm_list]
    print(f"Community embeddings: {_comm_embs.shape}")
else:
    _comm_embs  = np.zeros((0, 384), dtype=np.float32)
    _comm_dates = []
    print("No community summaries — global retrieval will return empty context.")

_STOP_ENTS = _FINANCIAL_STOPWORDS

def _valid_entity(name: str) -> bool:
    n = name.strip()
    if len(n) < 4:                          return False
    if n.lower() in _STOP_ENTS:             return False
    if not re.search(r"[A-Za-z]", n):       return False
    if not re.search(r"[A-Z]", n):          return False
    words = n.lower().split()
    if len(words) > 1 and all(w in _STOP_ENTS for w in words): return False
    return True

# ─────────────────────────────────────────────────────────────────────────────
# Temporal utilities
# ─────────────────────────────────────────────────────────────────────────────
def _parse_date_safe(ds: Optional[str]) -> Optional[datetime]:
    """Parse an ISO date string; return None on failure."""
    if not ds:
        return None
    try:
        return datetime.fromisoformat(ds[:10])
    except Exception:
        return None

def _date_score(edge_ts: Optional[str], query_ts: Optional[str],
                window_days: int = CFG["time_window_days"]) -> float:
    """Return a temporal relevance score in [0, 1].
    Score = 1.0 if within window, decays linearly to 0 at 2 window."""
    if not edge_ts or not query_ts:
        return 0.5           # unknown → neutral weight
    ed = _parse_date_safe(edge_ts)
    qd = _parse_date_safe(query_ts)
    if ed is None or qd is None:
        return 0.5
    delta = abs((ed - qd).days)
    if delta <= window_days:
        return 1.0
    if delta >= 2 * window_days:
        return 0.0
    return 1.0 - (delta - window_days) / window_days

def _extract_query_date(question: str) -> Optional[str]:
    """Heuristically extract a date anchor from the question text."""
    # Year
    m = re.search(r'\b(20[0-9]{2})\b', question)
    if m:
        return f"{m.group(1)}-06-15"   # mid-year as anchor
    # Quarter
    m = re.search(r'Q([1-4])\s+(20[0-9]{2})', question, re.IGNORECASE)
    if m:
        q, yr = int(m.group(1)), int(m.group(2))
        mo = q * 3 - 1
        return f"{yr}-{mo:02d}-15"
    return None

# ─────────────────────────────────────────────────────────────────────────────
# Local retrieval — with temporal ranking
# ─────────────────────────────────────────────────────────────────────────────
def local_retrieval(question: str, hop: int = CFG["hop"]) -> str:
    q_lower    = question.lower()
    query_date = _extract_query_date(question)

    # Entity matching (two passes)
    matched = [
        cid for cid, attrs in entity_registry.items()
        if _valid_entity(attrs["name"]) and attrs["name"].lower() in q_lower
    ]
    if not matched:
        q_words = set(re.findall(r'\b[A-Z][A-Za-z]{2,}\b', question))
        matched = [
            cid for cid, attrs in entity_registry.items()
            if _valid_entity(attrs["name"])
            and any(w.lower() in attrs["name"].lower() or
                    attrs["name"].lower() in w.lower() for w in q_words)
        ]
    matched = matched[:5]
    if not matched:
        return ""

    # BFS k-hop subgraph
    sub = set(matched)
    frontier = set(matched)
    for _ in range(hop):
        nxt = set()
        for node in frontier:
            if G_knowledge.has_node(node):
                nxt.update(G_knowledge.predecessors(node))
                nxt.update(G_knowledge.successors(node))
        sub.update(nxt)
        frontier = nxt - sub

    # Collect edges with temporal scores
    scored_lines = []
    for u, v, d in G_knowledge.edges(data=True):
        if u not in sub or v not in sub:
            continue
        un  = entity_registry.get(u, {}).get("name", u)
        vn  = entity_registry.get(v, {}).get("name", v)
        rt  = d.get("relation_type", "REL")
        ts  = d.get("time_interval", "")
        cf  = d.get("confidence", 0.0)
        try:
            sent = json.loads(d.get("provenance","{}")).get("sentence","")[:80]
        except Exception:
            sent = ""
        t_score = _date_score(ts, query_date)
        line = f"({un}) -[{rt}]-> ({vn})"
        if ts:   line += f" [{ts}]"
        if cf:   line += f" conf={cf:.2f}"
        if sent: line += f' | \"{sent}\"'
        # Composite score: relevance × temporal weight
        relevance = cf if cf else 0.5
        scored_lines.append((t_score * 0.4 + relevance * 0.6, line))

    # Sort by combined score descending, take top 40
    scored_lines.sort(key=lambda x: -x[0])
    lines = [l for _, l in scored_lines[:40]]
    return "\n".join(lines) if lines else (
        "Entities found: " + ", ".join(
            entity_registry.get(n,{}).get("name",n) for n in matched))

# ─────────────────────────────────────────────────────────────────────────────
# Global retrieval — with temporal filtering of community summaries
# ─────────────────────────────────────────────────────────────────────────────
def global_retrieval(question: str, k: int = CFG["top_k_global"]) -> str:
    if _comm_embs.shape[0] == 0:
        return ""
    query_date = _extract_query_date(question)
    qv   = embedder.encode([question], normalize_embeddings=True)[0].astype(np.float32)
    sims = _comm_embs @ qv                        # (N,)

    # Boost score for temporally relevant communities
    t_boosts = np.array([
        _date_score(d, query_date) for d in _comm_dates
    ], dtype=np.float32)
    combined = 0.7 * sims + 0.3 * t_boosts       # weighted combination

    idxs = np.argsort(-combined)[:k]
    parts = []
    for i in idxs:
        if i < len(_comm_list):
            c = _comm_list[int(i)]
            ts_str = c.get("earliest_date","")
            parts.append(
                f"[Community {c['community_id']} | {c['n_entities']} entities"
                f"{' | ' + ts_str if ts_str else ''}]\n{c['summary']}"
            )
    return "\n\n".join(parts)

# ─────────────────────────────────────────────────────────────────────────────
# Hybrid + router
# ─────────────────────────────────────────────────────────────────────────────
def hybrid_retrieval(question: str) -> str:
    parts = []
    lc = local_retrieval(question)
    gc = global_retrieval(question)
    if lc: parts.append(f"=== Local Subgraph ===\n{lc}")
    if gc: parts.append(f"=== Global Communities ===\n{gc}")
    return "\n\n".join(parts)

def route_and_retrieve(question: str) -> Tuple[str, str]:
    qt  = classify_query(question)
    ctx = (local_retrieval(question)   if qt == "local"
           else global_retrieval(question) if qt == "global"
           else hybrid_retrieval(question))
    return ctx, qt

print(f"Retrieval ready — temporal window: ±{CFG['time_window_days']} days.")

## 14. Answer Generation — Mistral-NeMo 12B

The model is instructed to answer **only from the provided context** and to say
"I don't know" when the context does not support an answer.
For conversational follow-ups, the question is first rewritten as standalone.

In [ ]:
_REWRITE_PROMPT = """Given the conversation history and a follow-up question,
rewrite the follow-up as a fully standalone question.

History:
{history}

Follow-up: {question}

Standalone question:"""

_ANSWER_PROMPT = """You are a financial analyst with access to retrieved Bloomberg news excerpts.
Answer the question using the provided context. Be specific and cite relevant facts.
If the context does not contain enough information, say so explicitly.

Retrieved context:
{context}

Question: {question}

Answer:"""

def generate_answer(
    question: str,
    context: str,
    history: Optional[List[Dict]] = None,
    run_safety: bool = CFG["safety_enabled"],
) -> Dict:
    """Generate answer with Mistral-NeMo; runs hallucination detection and Llama Guard.

    Returns a dict with keys:
      answer, hallucination_detected, unsupported_claims, safe, safety_categories
    """
    if history:
        hist_str = "\n".join(f"Q: {h['question']}\nA: {h['answer']}" for h in history[-3:])
        question = ollama(CFG["answer_model"],
                          _REWRITE_PROMPT.format(history=hist_str, question=question),
                          timeout=60)
    ctx = context.strip() or "No relevant information found in the knowledge graph."
    answer_text = ollama(CFG["answer_model"],
                         _ANSWER_PROMPT.format(context=ctx[:3000], question=question),
                         timeout=90)

    # Hallucination check
    halluc = detect_hallucination(answer_text, ctx)

    # Safety check (Llama Guard)
    safety = llama_guard_check(answer_text) if run_safety else {"safe": True, "categories": []}

    return {
        "answer"               : answer_text,
        "hallucination_detected": halluc["hallucination_detected"],
        "unsupported_claims"   : halluc["unsupported_claims"],
        "safe"                 : safety["safe"],
        "safety_categories"    : safety["categories"],
    }

print(f"Answer generator ready (model: {CFG['answer_model']}, safety={CFG['safety_enabled']})")

## 15. Load QA Set & Run Full Pipeline

Load the shared `qa_set.json` (generated in Notebook 1) and run all 20 questions
through the complete GraphRAG pipeline.

## 15b. Stratified QA Sampling — Graph Hierarchy Coverage

Before running the evaluation, we apply **stratified sampling** across three hierarchy tiers
to ensure the QA set exercises every level of the knowledge graph:

| Tier | Source | Selection |
|------|--------|-----------|
| **Leaf** | Individual high-degree entity nodes | Questions about specific entities |
| **Community** | Leiden community representatives | Questions about financial themes |
| **Cross-community** | Bridge edges between communities | Questions spanning multiple topics |

In [ ]:
import collections, random

def stratified_qa_sample(
    qa_set: List[Dict],
    G: "nx.MultiDiGraph",
    community_membership: Dict[str, int],
    n_per_tier: int = 7,
) -> List[Dict]:
    """Re-order / weight the QA set so evaluation covers all graph hierarchy levels.

    Assigns each QA item a tier based on which part of the graph its topic entity
    falls in, then samples up to n_per_tier from each tier.
    Remaining items fill up to the original size."""
    if not qa_set:
        return qa_set

    # Degree lookup
    degree = dict(G.degree())

    # Tier assignment heuristic
    leaf_ids, community_ids, bridge_ids, other_ids = [], [], [], []
    comm_sizes = collections.Counter(community_membership.values())

    for i, qa in enumerate(qa_set):
        # Try to match question to an entity in the graph
        q_lower = qa["question"].lower()
        matched_cids = [
            community_membership.get(cid)
            for cid, attrs in entity_registry.items()
            if _valid_entity(attrs["name"]) and attrs["name"].lower() in q_lower
        ]
        matched_cids = [c for c in matched_cids if c is not None]

        if not matched_cids:
            other_ids.append(i)
        elif len(set(matched_cids)) > 1:
            bridge_ids.append(i)     # spans multiple communities
        else:
            cid  = matched_cids[0]
            # Leaf: small community (≤3 members) or high-degree entity node
            relevant_nodes = [n for n,c in community_membership.items() if c==cid]
            max_deg = max((degree.get(n,0) for n in relevant_nodes), default=0)
            if comm_sizes[cid] <= 3 or max_deg >= 10:
                leaf_ids.append(i)
            else:
                community_ids.append(i)

    # Sample from each tier
    random.seed(42)
    def _sample(lst, n): return random.sample(lst, min(n, len(lst)))

    sampled_ids = (
        _sample(leaf_ids,      n_per_tier) +
        _sample(community_ids, n_per_tier) +
        _sample(bridge_ids,    n_per_tier)
    )

    # Fill up to original length with remaining
    remaining = [i for i in range(len(qa_set)) if i not in set(sampled_ids)]
    random.shuffle(remaining)
    final_ids = sampled_ids + remaining[: max(0, len(qa_set) - len(sampled_ids))]

    # Log tier breakdown
    tier_map = {i: "leaf" for i in leaf_ids}
    tier_map.update({i: "community" for i in community_ids})
    tier_map.update({i: "bridge" for i in bridge_ids})
    print(f"Stratified QA — leaf={len(leaf_ids)}  community={len(community_ids)}  "
          f"bridge={len(bridge_ids)}  other={len(other_ids)}")

    result = [dict(qa_set[i], tier=tier_map.get(i,"other")) for i in final_ids]
    return result

In [ ]:
if not os.path.exists(CFG["qa_set_path"]):
    raise FileNotFoundError(
        f"{CFG['qa_set_path']} not found. Run Baseline 1 (Pure LLM) notebook first.")

with open(CFG["qa_set_path"]) as _f:
    qa_set = json.load(_f)

print(f"QA set loaded: {len(qa_set)} questions")
for qa in qa_set[:5]:
    print(f"  [{qa['qa_id']:2d}] ({qa['topic']:8s}) {qa['question'][:65]}")
    
# ── Apply stratification if community membership is available ────────────────
if "community_membership" in dir() and community_membership:
    qa_set_stratified = stratified_qa_sample(
        qa_set if "qa_set" in dir() else [],
        G_knowledge,
        community_membership,
    )
    print(f"Stratified QA set: {len(qa_set_stratified)} items")
else:
    qa_set_stratified = qa_set if "qa_set" in dir() else []
    print("community_membership not yet available — stratification skipped.")

# ── Use stratified QA set if available ───────────────────────────────────────
_run_set = qa_set_stratified if "qa_set_stratified" in dir() and qa_set_stratified else qa_set
print(f"Running on {len(_run_set)} questions "
      f"({'stratified' if _run_set is not qa_set else 'original'} order)")

results: List[Dict] = []
_t0 = time.time()

for qa in tqdm(_run_set, desc="GraphRAG QA"):
    _ts = time.perf_counter()
    ctx, qtype  = route_and_retrieve(qa["question"])
    ans_dict    = generate_answer(qa["question"], ctx)
    latency     = (time.perf_counter() - _ts) * 1000

    results.append({
        "qa_id"                : qa["qa_id"],
        "article_id"           : qa.get("article_id"),
        "topic"                : qa.get("topic",""),
        "tier"                 : qa.get("tier","other"),
        "question"             : qa["question"],
        "reference_answer"     : qa.get("reference_answer",""),
        "predicted_answer"     : ans_dict["answer"],
        "context_used"         : ctx[:600],
        "query_type"           : qtype,
        "latency_ms"           : round(latency, 1),
        "hallucination_detected": ans_dict["hallucination_detected"],
        "unsupported_claims"   : ans_dict["unsupported_claims"],
        "safe"                 : ans_dict["safe"],
        "safety_categories"    : ans_dict["safety_categories"],
        "method"               : "graphrag_v2",
    })

_elapsed = time.time() - _t0
print(f"\nDone: {len(results)} answers in {_elapsed:.0f}s ({_elapsed/len(results):.1f}s each)")

# Hallucination summary
_n_halluc = sum(1 for r in results if r["hallucination_detected"])
_n_unsafe = sum(1 for r in results if not r["safe"])
print(f"Hallucinations detected: {_n_halluc}/{len(results)} ({_n_halluc/len(results):.1%})")
print(f"Safety flags           : {_n_unsafe}/{len(results)}")

# Preview first result
_r0 = results[0]
print(f"\nQ : {_r0['question']}")
print(f"A : {_r0['predicted_answer'][:200]}")
print(f"Halluc: {_r0['hallucination_detected']}  Safe: {_r0['safe']}")

## 16. RAG Triad Evaluation — LLM-as-Judge (Llama 3.2 3B)

| Metric | Definition |
|--------|------------|
| **Faithfulness** | Every claim in the answer is supported by the retrieved context |
| **Answer Relevance** | Answer directly addresses the question |
| **Context Precision** | Fraction of retrieved context useful for answering |

In [ ]:
_FAITH = ("Rate faithfulness of the answer to the context (0.0-1.0).\n"
          "Faithfulness = every claim is supported by the context.\n\n"
          "Context: {context}\nAnswer: {answer}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")
_RELEV = ("Rate relevance of the answer to the question (0.0-1.0).\n\n"
          "Question: {query}\nAnswer: {answer}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")
_PREC  = ("Rate context precision (0.0-1.0).\n"
          "Precision = fraction of context useful for answering the question.\n\n"
          "Question: {query}\nContext: {context}\n\n"
          "Output ONLY JSON: {{\"score\": <float>, \"reason\": \"<one sentence>\"}}")

def parse_score(raw: str) -> float:
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return float(json.loads(m.group())["score"]) if m else 0.0
    except Exception:
        return 0.0

print(f"Evaluating {len(results)} answers with {CFG['llm_model']} judge ...")
for r in tqdm(results, desc="Triad eval"):
    ctx_p = r["context_used"][:600]
    faith = parse_score(ollama(CFG["llm_model"],
                               _FAITH.format(context=ctx_p, answer=r["predicted_answer"])))
    relev = parse_score(ollama(CFG["llm_model"],
                               _RELEV.format(query=r["question"], answer=r["predicted_answer"])))
    prec  = parse_score(ollama(CFG["llm_model"],
                               _PREC.format(query=r["question"], context=ctx_p)))
    r["faithfulness"]      = round(faith, 3)
    r["answer_relevance"]  = round(relev, 3)
    r["context_precision"] = round(prec,  3)
    r["triad_avg"]         = round((faith + relev + prec) / 3, 3)

print()
print("=" * 60)
print("GRAPHRAG — BENCHMARK RESULTS")
print("=" * 60)
print(f"{'Metric':28s}  {'Mean':>7}  {'Std':>7}")
print("-" * 45)
for _key in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
    _vals = [r[_key] for r in results]
    print(f"{_key:28s}  {np.mean(_vals):7.4f}  {np.std(_vals):7.4f}")
print(f"{'latency_ms (avg)':28s}  {np.mean([r['latency_ms'] for r in results]):7.1f}")

## 17. Save Results & Three-Way Comparison

In [ ]:
summary = {
    "method"             : "GraphRAG-v2 (FinBERT-SupervisedNER + BilinearRE + SSHG-HPO + Leiden + Mistral-Nemo)",
    "n_questions"        : len(results),
    "kg_nodes"           : G_knowledge.number_of_nodes(),
    "kg_edges"           : G_knowledge.number_of_edges(),
    "n_communities"      : len(community_store),
    # RAG Triad
    "mean_faithfulness"  : float(np.mean([r["faithfulness"]       for r in results])),
    "mean_relevance"     : float(np.mean([r["answer_relevance"]   for r in results])),
    "mean_ctx_precision" : float(np.mean([r["context_precision"]  for r in results])),
    "mean_triad_avg"     : float(np.mean([r["triad_avg"]          for r in results])),
    "mean_latency_ms"    : float(np.mean([r["latency_ms"]         for r in results])),
    # Quality / safety flags
    "hallucination_rate" : float(np.mean([r.get("hallucination_detected", False) for r in results])),
    "mean_dep_score"     : float(np.mean([r.get("dep_score", 0.0) for r in results])),
    # HPO params applied
    "hpo_lambda1"        : CFG["sshg_lambda1"],
    "hpo_lambda2"        : CFG["sshg_lambda2"],
    "hpo_lambda3"        : CFG["sshg_lambda3"],
    "hpo_rel_threshold"  : CFG["rel_confidence_threshold"],
    # Stratification breakdown
    "tier_counts"        : {t: sum(1 for r in results if r.get("tier")==t)
                            for t in ("leaf","community","bridge","other")},
    "per_question"       : results,
}

with open(CFG["results_path"], "w") as _f:
    json.dump(summary, _f, indent=2)
print(f"Saved: {CFG['results_path']} ({os.path.getsize(CFG['results_path'])//1024} KB)")

# ── Three-way comparison ───────────────────────────────────────────────────────
def _load_baseline(path):
    if not os.path.exists(path): return {}
    with open(path) as f: d = json.load(f)
    return {
        "faithfulness"     : d.get("mean_faithfulness",   d.get("mean_faith", 0)),
        "answer_relevance" : d.get("mean_relevance",      d.get("mean_answer_relevance", 0)),
        "context_precision": d.get("mean_ctx_precision",  d.get("mean_context_precision", 0)),
        "triad_avg"        : d.get("mean_triad_avg", 0),
    }

_llm  = _load_baseline("baseline_llm_results.json")
_rag  = _load_baseline("rag_results.json")
_grag = {
    "faithfulness"     : summary["mean_faithfulness"],
    "answer_relevance" : summary["mean_relevance"],
    "context_precision": summary["mean_ctx_precision"],
    "triad_avg"        : summary["mean_triad_avg"],
}

print()
print("=" * 68)
print("THREE-WAY BENCHMARK (GraphRAG-v2 vs Vanilla RAG vs Pure LLM)")
print("=" * 68)
_hdr = f"{'Metric':28s}  {'Pure LLM':>9}  {'Vanilla RAG':>11}  {'GraphRAG-v2':>11}"
print(_hdr)
print("-" * 68)
for _k in ["faithfulness","answer_relevance","context_precision","triad_avg"]:
    _lv  = f"{_llm.get(_k,0):.3f}"  if _llm  else "  —  "
    _rv  = f"{_rag.get(_k,0):.3f}"  if _rag  else "  —  "
    _gv  = f"{_grag[_k]:.3f}"
    print(f"  {_k:28s}  {_lv:>9}  {_rv:>11}  {_gv:>11}")
print()
print(f"  Hallucination rate   : {summary['hallucination_rate']:.1%}")
print(f"  Mean DEP score       : {summary['mean_dep_score']:.3f}")
print(f"  SSHG weights (tuned) : λ₁={summary['hpo_lambda1']:.3f}  "
      f"λ₂={summary['hpo_lambda2']:.3f}  λ₃={summary['hpo_lambda3']:.3f}")
print(f"  Relation threshold   : {summary['hpo_rel_threshold']:.3f}")
print(f"  Tier breakdown       : {summary['tier_counts']}")

## Summary

### GraphRAG-v2 — Component Inventory

| Component | Choice | Role |
|-----------|--------|------|
| **Encoder** | `ProsusAI/finbert` | Domain-adapted embeddings for financial text |
| **NER** | FinBERT fine-tuned on `nlpaueb/finer-139` (BIO head) | **Fully supervised** entity span detection; log-prob uncertainty filter |
| **Relation extraction** | Supervised **bilinear classifier** (trained on silver triples) | Replaces silver-label heuristics; bilinear scores used as uncertainty signal |
| **SSHG** | Syntax (λ₁) + Semantic (λ₂) + Co-occurrence (λ₃) weighted edges | Configurable edge weights; **Optuna HPO** optimises all three |
| **HPO** | Optuna TPE sampler | Tunes λ₁/λ₂/λ₃ + relation confidence threshold + semantic gate |
| **Graph propagation** | 2-layer GCN (normalised Laplacian) | Neighbourhood-smoothed entity representations |
| **Data quality** | Data Error Potential (DEP) filter | Drops noisy chunks before KG insertion |
| **Factuality** | Context-grounded LLM audit (5 % sample) | Verifies extracted relations against source sentences |
| **Hallucination** | LLM-as-judge on final answers | Flags unsupported claims before returning to user |
| **Safety** | Llama Guard (optional) | Bias / harm monitoring on generated answers |
| **Graph store** | Neo4j (+ NetworkX fallback) | Append-only property graph with temporal metadata |
| **Community detection** | Leiden algorithm | Hierarchical global retrieval index |
| **Retrieval** | Local (k-hop BFS) + Global (FAISS cosine) with **temporal ranking** | `time_interval` metadata used to boost temporally relevant edges & communities |
| **QA evaluation** | **Stratified sampling** across leaf / community / bridge tiers | Ensures coverage of all graph hierarchy levels |
| **Evaluation metrics** | RAG Triad (Faithfulness, Relevance, Context Precision) + Hallucination rate | LLM-as-judge + deterministic DEP score |

### Throughput Scaling

To scale beyond the default run size, increase `CFG["process_n_chunks"]` and relation-training data volume via `CFG["re_train_chunks"]`.
The SSHG → GCN → RE steps are chunk-parallel by design; only the final KG merge requires serialisation.